In [1]:
import os
import json

# --- Data Preparation ---
# Creating the puzzle files in the format required by the assignment
puzzles = {
    "easy.txt": [
        "004030050", "609400000", "005100489",
        "000060930", "300807002", "026040000",
        "453009600", "000004705", "090050200",
    ],
    "medium.txt": [
        "530070000", "600195000", "098000060",
        "800060003", "400803001", "700020006",
        "060000280", "000419005", "000080079",
    ],
    "hard.txt": [
        "400000805", "030000000", "000700000",
        "020000060", "000080400", "000010000",
        "000603070", "500200000", "104000000",
    ],
    "veryhard.txt": [
        "100007090", "030020008", "009600500",
        "005300900", "010080002", "600004000",
        "300000010", "040000007", "007000300",
    ],
}

# Write dictionary data into physical .txt files
for filename, lines in puzzles.items():
    with open(filename, "w") as f:
        f.write("\n".join(lines))

class SudokuSolver:
    def __init__(self, filename: str):
        """Initialize the solver, load the board, and set up CSP variables."""
        self.board = []
        with open(filename) as f:
            for line in f:
                line = line.strip()
                if line:
                    self.board.append([int(ch) for ch in line])

        # Performance metrics required for the report
        self.backtrack_calls = 0
        self.backtrack_failures = 0
        
        # Pre-calculate neighbors (peers) for every cell to speed up constraints
        self.neighbors = {
            (r, c): self._compute_neighbors(r, c)
            for r in range(9) for c in range(9)
        }
        
        # Initialize domains for each cell (1-9 for empty, or fixed value)
        self.domains = self._initialize_domains()

    def _compute_neighbors(self, r: int, c: int) -> set:
        """Finds all cells in the same row, column, and 3x3 box."""
        peers = set()
        for i in range(9):
            peers.add((r, i)) # Row peers
            peers.add((i, c)) # Column peers
        
        # 3x3 Box peers
        box_r, box_c = (r // 3) * 3, (c // 3) * 3
        for dr in range(3):
            for dc in range(3):
                peers.add((box_r + dr, box_c + dc))
        
        peers.discard((r, c)) # Remove the cell itself from neighbors
        return peers

    def _initialize_domains(self) -> dict:
        """Initializes the possible values for each cell (the 'Domain')."""
        domains = {}
        for r in range(9):
            for c in range(9):
                if self.board[r][c] != 0:
                    # If fixed, domain is just that number
                    domains[(r, c)] = [self.board[r][c]]
                else:
                    # If empty, check 1-9 against current board constraints
                    domains[(r, c)] = [
                        v for v in range(1, 10)
                        if self._is_consistent((r, c), v)
                    ]
        return domains

    def ac3(self, queue: list | None = None) -> bool:
        """
        AC-3 Algorithm: Enforces Arc Consistency. 
        Prunes domains to ensure no cell has an impossible value.
        """
        if queue is None:
            # If no queue provided, check every arc in the puzzle
            queue = [
                (xi, xj)
                for xi in self.domains
                for xj in self.neighbors[xi]
            ]

        while queue:
            xi, xj = queue.pop(0)
            if self._revise(xi, xj):
                # If domain of xi is empty, the puzzle is unsolvable in this state
                if not self.domains[xi]:
                    return False
                # If xi was changed, we must re-check its neighbors
                for xk in self.neighbors[xi]:
                    if xk != xj:
                        queue.append((xk, xi))
        return True

    def _revise(self, xi: tuple, xj: tuple) -> bool:
        """Checks if xi's domain needs to be pruned based on xj's domain."""
        revised = False
        for x in self.domains[xi][:]:
            # If there is no value in xj's domain different from x, prune x
            if not any(y != x for y in self.domains[xj]):
                self.domains[xi].remove(x)
                revised = True
        return revised

    def _is_consistent(self, var: tuple, val: int) -> bool:
        """Checks if placing 'val' at 'var' violates Sudoku rules."""
        r, c = var
        for nr, nc in self.neighbors[var]:
            if self.board[nr][nc] == val:
                return False
        return True

    def solve(self) -> bool:
        """Starts the solving process using AC-3 pre-processing then Backtracking."""
        self.ac3() # Initial pruning
        return self._backtrack()

    def _backtrack(self) -> bool:
        """The core recursive search function with Forward Checking."""
        self.backtrack_calls += 1
        
        # Find all empty cells
        unassigned = [
            (r, c)
            for r in range(9) for c in range(9)
            if self.board[r][c] == 0
        ]
        
        # Base Case: If no unassigned cells, we are done!
        if not unassigned:
            return True

        # MRV Heuristic: Choose the cell with the fewest possible values left
        var = min(unassigned, key=lambda v: len(self.domains[v]))
        r, c = var

        for val in list(self.domains[var]):
            if self._is_consistent(var, val):
                # Try assigning value
                self.board[r][c] = val
                saved_domains = {k: list(v) for k, v in self.domains.items()}
                self.domains[var] = [val]

                # Forward Checking: Use AC-3 to prune neighbors based on this assignment
                arcs = [(xk, var) for xk in self.neighbors[var]]
                if self.ac3(arcs):
                    if self._backtrack():
                        return True

                # Undo assignment (Backtrack) if the path failed
                self.board[r][c] = 0
                self.domains = saved_domains

        self.backtrack_failures += 1
        return False

    def print_board(self):
        """Helper to display the Sudoku board in a readable grid."""
        for i in range(9):
            if i % 3 == 0 and i != 0:
                print("- - - - - - - - - - - -")
            for j in range(9):
                if j % 3 == 0 and j != 0:
                    print("| ", end="")
                print(self.board[i][j], end=" ")
            print()

# --- Main Execution ---
for filename in ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]:
    print(f"\n{'='*10} Solving: {filename} {'='*10}")
    solver = SudokuSolver(filename)
    if solver.solve():
        solver.print_board()
        print(f"\nBacktrack Calls: {solver.backtrack_calls}")
        print(f"Backtrack Failures: {solver.backtrack_failures}")
    else:
        print("No solution found.")


========== Solving: easy.txt ==========
7 8 4 | 9 3 2 | 1 5 6 
6 1 9 | 4 8 5 | 3 2 7 
2 3 5 | 1 7 6 | 4 8 9 
- - - - - - - - - - - -
5 7 8 | 2 6 1 | 9 3 4 
3 4 1 | 8 9 7 | 5 6 2 
9 2 6 | 5 4 3 | 8 7 1 
- - - - - - - - - - - -
4 5 3 | 7 2 9 | 6 1 8 
8 6 2 | 3 1 4 | 7 9 5 
1 9 7 | 6 5 8 | 2 4 3 

Backtrack Calls: 50
Backtrack Failures: 0

========== Solving: medium.txt ==========
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
- - - - - - - - - - - -
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 7 9 1 
7 1 3 | 9 2 4 | 8 5 6 
- - - - - - - - - - - -
9 6 1 | 5 3 7 | 2 8 4 
2 8 7 | 4 1 9 | 6 3 5 
3 4 5 | 2 8 6 | 1 7 9 

Backtrack Calls: 52
Backtrack Failures: 0

========== Solving: hard.txt ==========
4 1 7 | 3 6 9 | 8 2 5 
6 3 2 | 1 5 8 | 9 4 7 
9 5 8 | 7 2 4 | 3 1 6 
- - - - - - - - - - - -
8 2 5 | 4 3 7 | 1 6 9 
7 9 1 | 5 8 6 | 4 3 2 
3 4 6 | 9 1 2 | 7 5 8 
- - - - - - - - - - - -
2 8 9 | 6 4 3 | 5 7 1 
5 7 3 | 2 9 1 | 6 8 4 
1 6 4 | 8 7 5 | 2 9 3 

Backtrack Calls: 213
Ba